
# MiniMax H3 — Quality-First Kaggle API Server (Enterprise)

This notebook builds a **headless MiniMax H3 inference server** on Kaggle.

1. Authenticates to Hugging Face via Kaggle Secrets / env vars (never hardcodes keys).
2. Pins a tested WanGP commit; everything (models, caches, server) lives under `/tmp`.
3. Uses MiniMax H3 FL2VA pruned Rank-8 INT8 ConvRot + Qwen3-VL 32B Q4_K_M.
4. WanGP/MMGP Profile 4 + SDPA attention (T4-safe) + vmm_spill allocator.
5. Turbo / TeaCache / Spectrum / First-Block Cache OFF by default — quality first.
6. Real startup smoke test before the public endpoint is printed.
7. FastAPI server + Cloudflare Quick Tunnel; prints BASE URL + API KEY only when healthy.
8. Ships `h3_server.py` and `h3_cli.py` alongside this notebook.

**Kaggle secrets required:** add `HF_TOKEN` (and optionally `KAGGLE_KEY`) under *Add-ons → Secrets*.
The keys you shared in chat should be rotated — they were never written into this notebook.


In [ ]:

# 0) Credentials & cache locations.
# Priority: Kaggle Secrets -> environment variables -> local kaggle.json (for CLI use only).
# Tokens are never printed and never stored in the notebook.

import os, sys
from pathlib import Path

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    print("WARNING: HF_TOKEN not found. Set it in Kaggle Secrets or env before pulling gated assets.")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded (length %d)." % len(HF_TOKEN))

# Route all heavy caches/models to /tmp for better storage behavior on Kaggle.
# Disable HF xet downloads (often blocked on Kaggle) so downloads fall back to normal HTTPS.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_HF_TRANSFER", "1")
for var, path in {
    "HF_HOME": "/tmp/hf",
    "HF_HUB_CACHE": "/tmp/hf/hub",
    "TRANSFORMERS_CACHE": "/tmp/hf/transformers",
    "TORCH_HOME": "/tmp/torch",
    "XDG_CACHE_HOME": "/tmp/xdg_cache",
    "PIP_CACHE_DIR": "/tmp/pip_cache",
}.items():
    os.environ.setdefault(var, path)
    Path(path).mkdir(parents=True, exist_ok=True)

print("Cache roots:", {k: os.environ[k] for k in ("HF_HOME", "TORCH_HOME", "XDG_CACHE_HOME")})


In [ ]:

# 1) Core configuration — edit only the USER CONFIG block if needed.
# Values can also be overridden by environment variables for reproducibility.

from pathlib import Path
import os, json, secrets, subprocess, sys, hashlib, shutil, re, time

# ---------------- USER CONFIG ----------------
QUALITY = os.environ.get("H3_QUALITY", "maximum")

DEFAULT_RESOLUTION = os.environ.get("H3_RESOLUTION", "1344x768")
DEFAULT_STEPS = int(os.environ.get("H3_STEPS", "28"))
DEFAULT_DURATION_SECONDS = float(os.environ.get("H3_DURATION", "5.0"))
DEFAULT_SEED = int(os.environ.get("H3_SEED", "-1"))

STARTUP_SMOKE_TEST = os.environ.get("H3_STARTUP_SMOKE_TEST", "1") == "1"

WAN2GP_REPO = "https://github.com/deepbeepmeep/Wan2GP.git"
WAN2GP_COMMIT = os.environ.get("WAN2GP_COMMIT", "ec9566a6593727a922e516fbd0919f9b9210c949")

HF_REPO = "DeepBeepMeep/MiniMax-H3"
H3_FL2VA_REV = "fec7846aef352e58a1cfb699455e3d104281e68b"
H3_FL2VA_FILE = "MiniMax-H3-FL2VA-pruned_rank8_int8_convrot.safetensors"

QWEN_REV = "d0ca4f71576145f2daa3002e0d33b97d822200a9"
QWEN_Q4_FILE = "qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf"

CLOUDFLARED_VERSION = "2026.10.0"

# ---------------- PATHS (all under /tmp) ----------------
WAN_ROOT = Path("/tmp/Wan2GP")
MODEL_ROOT = Path("/tmp/h3_models")
API_ROOT = Path("/tmp/h3_api")
OUTPUT_ROOT = API_ROOT / "outputs"
UPLOAD_ROOT = API_ROOT / "uploads"
LOG_ROOT = API_ROOT / "logs"

API_PORT = 8000
API_HOST = "127.0.0.1"

API_KEY = os.environ.get("H3_API_KEY") or ("h3_" + secrets.token_urlsafe(32))

for p in (MODEL_ROOT, API_ROOT, OUTPUT_ROOT, UPLOAD_ROOT, LOG_ROOT):
    p.mkdir(parents=True, exist_ok=True)

print("Configuration prepared.")
print(f"WanGP commit:        {WAN2GP_COMMIT}")
print(f"H3 checkpoint:       {H3_FL2VA_FILE}")
print(f"H3 revision:         {H3_FL2VA_REV}")
print(f"Qwen encoder:        {QWEN_Q4_FILE}")
print(f"Default resolution:  {DEFAULT_RESOLUTION}")
print(f"Default steps:       {DEFAULT_STEPS}")
print(f"Startup smoke test:  {STARTUP_SMOKE_TEST}")


In [ ]:

# 2) Resource preflight — do not proceed silently if storage or GPU conditions are unsafe.

import psutil
import torch

def run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True, check=False)

gpu = run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"])
print("GPU inventory:")
print(gpu.stdout.strip() or gpu.stderr.strip())

ram = psutil.virtual_memory()
tmp_free = shutil.disk_usage("/tmp").free
work_free = shutil.disk_usage("/kaggle/working").free if Path("/kaggle/working").exists() else 0

print(f"\nSystem RAM:          {ram.total/(1024**3):.1f} GB total / {ram.available/(1024**3):.1f} GB available")
print(f"/tmp free:           {tmp_free/(1024**3):.1f} GB")
print(f"/kaggle/working free:{work_free/(1024**3):.1f} GB")

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is visible. Enable a Kaggle GPU accelerator.")

gpu_count = torch.cuda.device_count()
print(f"CUDA devices visible: {gpu_count}")

if gpu_count > 1:
    print("\nNOTE: Multiple GPUs visible. Stable nested path uses a single GPU; second GPU is not assumed to add VRAM.")

if tmp_free < 15 * (1024**3):
    raise RuntimeError(f"Less than 15 GB free on /tmp — free space or reduce the model set. Currently {tmp_free/(1024**3):.1f} GB.")


In [ ]:

# 3) Clone and PIN WanGP instead of tracking a mutable main branch.

if not WAN_ROOT.exists():
    subprocess.run(["git", "clone", WAN2GP_REPO, str(WAN_ROOT)], check=True)

subprocess.run(["git", "-C", str(WAN_ROOT), "fetch", "--all", "--tags", "--prune"], check=True)
subprocess.run(["git", "-C", str(WAN_ROOT), "checkout", "--force", WAN2GP_COMMIT], check=True)

actual = subprocess.check_output(["git", "-C", str(WAN_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("WanGP pinned commit:", actual)
if actual != WAN2GP_COMMIT:
    raise RuntimeError(f"Unexpected WanGP commit: {actual}")


In [ ]:

# 4) Install dependencies, authenticate Hugging Face, and verify the stack.

print("Python:", sys.version)
try:
    import torch as _torch
    print("Torch:", _torch.__version__, "| CUDA:", _torch.version.cuda)
except Exception as exc:
    raise RuntimeError(f"PyTorch import failed: {exc}")

if HF_TOKEN:
    try:
        subprocess.run([sys.executable, "-m", "huggingface_hub.cli.login", "--token", HF_TOKEN, "--add-to-git-credential"], check=False, capture_output=True)
    except Exception:
        pass
    try:
        subprocess.run(["hf", "auth", "login", "--token", HF_TOKEN], check=False, capture_output=True)
    except Exception:
        pass
    print("Hugging Face authentication configured.")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(WAN_ROOT / "requirements.txt")], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "fastapi>=0.115,<1", "uvicorn>=0.34,<1", "python-multipart>=0.0.20,<1",
     "requests>=2.32,<3", "psutil>=6,<8"],
    check=True,
)
print("Dependencies installed.")



## 5) Build a reproducible H3 model definition

The current WanGP H3 handler knows the pruned H3 architecture and the official Turbo LoRA, but its generic defaults also expose older/full checkpoints.

For this notebook we create a **local finetune definition** that pins exactly:

- FL2VA pruned Rank-8 INT8 ConvRot
- Qwen3-VL 32B Q4_K_M GGUF
- H3's normal audio/video VAE assets
- the fixed Hugging Face revisions above

This keeps the notebook from silently drifting to a different model file.


In [ ]:

# Write local, pinned H3 finetune definitions before the WanGP API runtime is initialized.

FINETUNE_DIR = WAN_ROOT / "finetunes"
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

H3_URL = f"https://huggingface.co/{HF_REPO}/resolve/main/{H3_FL2VA_FILE}"
QWEN_URL = f"https://huggingface.co/{HF_REPO}/resolve/main/Qwen3-VL-32B-Instruct/{QWEN_Q4_FILE}"

fl2va_def = {
    "model": {
        "name": "MiniMax H3 FL2VA — Pruned Rank-8 INT8 ConvRot + Q4",
        "architecture": "minimax_h3_fl2va_pruned",
        "description": (
            "Pinned quality-first MiniMax H3 FL2VA definition for Kaggle. "
            "Pruned Rank-8 INT8 ConvRot transformer and Qwen3-VL 32B Q4_K_M."
        ),
        "URLs": [H3_URL],
        "text_encoder_URLs": [QWEN_URL],
        "text_encoder_variant": "gguf_q4_k_m",
        "video_vae_file": "MiniMax-H3-video_vae_fp16.safetensors",
        "audio_vae_file": "MiniMax-H3-audio_vae_fp32.safetensors",
        "infos": (
            "Pinned to a specific MiniMax-H3 revision. "
            "Quality-first defaults: no Turbo, no TeaCache, no Spectrum, no First Block Cache."
        ),
    }
}

REF_REV = H3_FL2VA_REV
REF_FILE = "MiniMax-H3-Ref2VA-pruned_rank8_int8_convrot.safetensors"
REF_URL = f"https://huggingface.co/{HF_REPO}/resolve/main/{REF_FILE}"

ref2va_def = {
    "model": {
        "name": "MiniMax H3 Ref2VA — Pruned Rank-8 INT8 ConvRot + Q4",
        "architecture": "minimax_h3_ref2va_pruned",
        "description": "Pinned MiniMax H3 Ref2VA for character/appearance/motion references.",
        "URLs": [REF_URL],
        "text_encoder_URLs": [QWEN_URL],
        "text_encoder_variant": "gguf_q4_k_m",
        "video_vae_file": "MiniMax-H3-video_vae_fp16.safetensors",
        "audio_vae_file": "MiniMax-H3-audio_vae_fp32.safetensors",
        "infos": "Pinned to a specific MiniMax-H3 revision.",
    }
}

(WAN_ROOT / "finetunes" / "h3_kaggle_fl2va_rank8_q4.json").write_text(json.dumps(fl2va_def, indent=2), encoding="utf-8")
(WAN_ROOT / "finetunes" / "h3_kaggle_ref2va_rank8_q4.json").write_text(json.dumps(ref2va_def, indent=2), encoding="utf-8")

print("Wrote pinned model definitions:")
print(WAN_ROOT / "finetunes" / "h3_kaggle_fl2va_rank8_q4.json")
print(WAN_ROOT / "finetunes" / "h3_kaggle_ref2va_rank8_q4.json")


In [ ]:

# 6) Download cloudflared (Cloudflare Quick Tunnel; no ngrok), pinned + version-verified.

cloudflared = API_ROOT / "cloudflared"
if not cloudflared.exists():
    url = f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
    subprocess.run(["curl", "-L", "--fail", "--retry", "5", "-o", str(cloudflared), url], check=True)
    cloudflared.chmod(0o755)

ver = subprocess.run([str(cloudflared), "--version"], capture_output=True, text=True, check=True)
print(ver.stdout.strip())
if CLOUDFLARED_VERSION not in ver.stdout:
    raise RuntimeError("cloudflared version mismatch")


In [ ]:
# 7) Write the headless FastAPI H3 server. No Gradio is started.
SERVER_PY = API_ROOT / "h3_server.py"
SERVER_SOURCE = r'''#!/usr/bin/env python3
from __future__ import annotations

import json
import os
import shutil
import sys
import threading
import time
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from fastapi import FastAPI, HTTPException, Request, UploadFile
from fastapi.responses import FileResponse, JSONResponse
import psutil
import torch
import uvicorn

WAN_ROOT = Path(os.environ["WAN2GP_ROOT"]).resolve()
OUTPUT_DIR = Path(os.environ.get("H3_OUTPUT_DIR", "/tmp/h3_api_outputs")).resolve()
UPLOAD_DIR = Path(os.environ.get("H3_UPLOAD_DIR", "/tmp/h3_api_uploads")).resolve()
MODEL_TYPE_FL2VA = os.environ["H3_MODEL_TYPE_FL2VA"]
MODEL_TYPE_REF2VA = os.environ.get("H3_MODEL_TYPE_REF2VA", "")
PORT = int(os.environ.get("H3_API_PORT", "8000"))
HOST = os.environ.get("H3_API_HOST", "127.0.0.1")
API_KEY = os.environ["H3_API_KEY"]
DEFAULT_RESOLUTION = os.environ.get("H3_DEFAULT_RESOLUTION", "1344x768")
DEFAULT_STEPS = int(os.environ.get("H3_DEFAULT_STEPS", "28"))
DEFAULT_DURATION = float(os.environ.get("H3_DEFAULT_DURATION", "5.0"))
DEFAULT_SEED = int(os.environ.get("H3_DEFAULT_SEED", "-1"))
STARTUP_SMOKE_TEST = os.environ.get("H3_STARTUP_SMOKE_TEST", "1") == "1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WAN_ROOT))

from shared.api import init  # noqa: E402


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def require_auth(request: Request) -> None:
    auth = request.headers.get("authorization", "")
    if not auth.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing Bearer token")
    if auth[7:].strip() != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")


def gpu_snapshot() -> list[dict[str, Any]]:
    rows = []
    if not torch.cuda.is_available():
        return rows
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        free_b, total_b = torch.cuda.mem_get_info(idx)
        rows.append({
            "index": idx,
            "name": props.name,
            "total_vram_gb": round(total_b / 2**30, 2),
            "free_vram_gb": round(free_b / 2**30, 2),
            "compute_capability": f"{props.major}.{props.minor}",
        })
    return rows


def normalize_resolution(value: str | None) -> str:
    value = str(value or DEFAULT_RESOLUTION).strip().lower().replace(" ", "")
    if "x" not in value:
        raise ValueError("resolution must be WIDTHxHEIGHT")
    w, h = value.split("x", 1)
    try:
        wi, hi = int(w), int(h)
    except ValueError as exc:
        raise ValueError("resolution must contain integer dimensions") from exc
    if wi < 256 or hi < 256:
        raise ValueError("resolution is too small")
    if wi % 32 or hi % 32:
        raise ValueError("width and height must be multiples of 32")
    if wi * hi > 2_200_000:
        raise ValueError("resolution exceeds the conservative 2.2 MP Kaggle safety limit")
    return f"{wi}x{hi}"


def set_if_present(settings: dict[str, Any], key: str, value: Any) -> None:
    if key in settings:
        settings[key] = value


@dataclass
class JobRecord:
    id: str
    status: str
    created_at: str
    started_at: str | None = None
    finished_at: str | None = None
    request: dict[str, Any] | None = None
    model_type: str | None = None
    generated_files: list[str] | None = None
    primary_file: str | None = None
    error: str | None = None


SESSION = None
JOB_LOCK = threading.Lock()
JOBS: dict[str, JobRecord] = {}
ACTIVE_JOB_ID: str | None = None
READY = False
STARTUP_ERROR: str | None = None
SMOKE_OUTPUT: str | None = None


def model_type_from_public_name(name: str | None) -> str:
    n = str(name or "fl2va").strip().lower()
    if n in {"fl2va", "h3", "h3-fl2va"}:
        return MODEL_TYPE_FL2VA
    if n in {"ref2va", "reference", "h3-ref2va"}:
        if not MODEL_TYPE_REF2VA:
            raise ValueError("Ref2VA is not enabled")
        return MODEL_TYPE_REF2VA
    raise ValueError("model must be fl2va or ref2va")


def pick_primary(files: list[str]) -> str | None:
    for p in files:
        if str(p).lower().endswith((".mp4", ".mov", ".webm", ".mkv")) and Path(p).exists():
            return str(Path(p).resolve())
    for p in files:
        if Path(p).exists():
            return str(Path(p).resolve())
    return None


def build_settings(payload: dict[str, Any], media: dict[str, Any]) -> tuple[str, dict[str, Any]]:
    model_type = model_type_from_public_name(payload.get("model", "fl2va"))
    settings = SESSION.get_default_settings(model_type)
    settings["model_type"] = model_type

    prompt = str(payload.get("prompt", "")).strip()
    if not prompt:
        raise ValueError("prompt is required")
    settings["prompt"] = prompt

    set_if_present(settings, "resolution", normalize_resolution(payload.get("resolution")))
    set_if_present(settings, "duration_seconds", float(payload.get("duration_seconds", DEFAULT_DURATION)))
    set_if_present(settings, "num_inference_steps", int(payload.get("steps", DEFAULT_STEPS)))
    set_if_present(settings, "seed", int(payload.get("seed", DEFAULT_SEED)))

    quality = str(payload.get("quality", "maximum")).lower()
    if quality not in {"maximum", "max", "quality"}:
        raise ValueError("This quality-first server only exposes quality='maximum'.")

    sampler = payload.get("sampler")
    if sampler:
        sampler = str(sampler).lower()
        if sampler not in {"euler", "res_multistep", "ralston_2s"}:
            raise ValueError("sampler must be euler, res_multistep, or ralston_2s")
        for key in ("sample_solver", "sampler", "scheduler"):
            set_if_present(settings, key, sampler)

    for key in (
        "image_start", "image_end", "image_refs",
        "video_guide", "video_guide2", "video_guide3",
        "audio_guide", "audio_guide2", "audio_guide3",
    ):
        if media.get(key):
            settings[key] = media[key]

    return model_type, settings


async def save_upload(upload: UploadFile, prefix: str) -> str:
    name = Path(upload.filename or "upload.bin").name
    dest = UPLOAD_DIR / f"{prefix}_{uuid.uuid4().hex}_{name}"
    with dest.open("wb") as f:
        shutil.copyfileobj(upload.file, f)
    return str(dest.resolve())


async def parse_input(request: Request) -> tuple[dict[str, Any], dict[str, Any]]:
    content_type = request.headers.get("content-type", "").lower()
    media: dict[str, Any] = {}
    if content_type.startswith("multipart/form-data"):
        form = await request.form()
        raw = form.get("payload") or form.get("request") or "{}"
        payload = json.loads(str(raw))

        mapping = {
            "image_start": "image_start",
            "image_end": "image_end",
            "video_guide": "video_guide",
            "video_guide2": "video_guide2",
            "video_guide3": "video_guide3",
            "audio_guide": "audio_guide",
            "audio_guide2": "audio_guide2",
            "audio_guide3": "audio_guide3",
        }
        for field, out_name in mapping.items():
            item = form.get(field)
            if isinstance(item, UploadFile):
                media[out_name] = await save_upload(item, field)

        refs = []
        getlist = getattr(form, "getlist", None)
        if callable(getlist):
            for item in getlist("reference_image"):
                if isinstance(item, UploadFile):
                    refs.append(await save_upload(item, "ref"))
        elif isinstance(form.get("reference_image"), UploadFile):
            refs.append(await save_upload(form["reference_image"], "ref"))
        if refs:
            media["image_refs"] = refs
        return payload, media

    body = await request.body()
    payload = json.loads(body.decode("utf-8") or "{}")
    return payload, media


def run_job(job_id: str, settings: dict[str, Any]) -> None:
    global ACTIVE_JOB_ID
    record = JOBS[job_id]
    record.status = "running"
    record.started_at = utc_now()
    try:
        job = SESSION.submit_task(settings)
        result = job.result(timeout=None)
        files = [str(Path(p).resolve()) for p in (result.generated_files or []) if Path(p).exists()]
        record.generated_files = files
        record.primary_file = pick_primary(files)
        record.status = "succeeded" if result.success else "failed"
        record.error = None if result.success else (
            "\n".join(getattr(e, "message", str(e)) for e in (result.errors or [])) or "Generation failed"
        )
    except Exception as exc:
        record.status = "failed"
        record.error = f"{type(exc).__name__}: {exc}"
    finally:
        record.finished_at = utc_now()
        with JOB_LOCK:
            ACTIVE_JOB_ID = None


def startup_smoke() -> None:
    global SMOKE_OUTPUT
    if not STARTUP_SMOKE_TEST:
        return
    settings = SESSION.get_default_settings(MODEL_TYPE_FL2VA)
    settings["model_type"] = MODEL_TYPE_FL2VA
    settings["prompt"] = (
        "integrated_multimodal_description: [Shot 1] A cinematic single take in a "
        "rainy glass observatory at night. A scientist turns toward a glowing console "
        "with a small natural smile; subtle slow camera push; realistic materials, "
        "stable anatomy and physically coherent motion. overall_soundscape: rain on "
        "glass, low electrical hum, one clean relay click, quiet room tone."
    )
    set_if_present(settings, "resolution", DEFAULT_RESOLUTION)
    set_if_present(settings, "duration_seconds", 4.5)
    set_if_present(settings, "num_inference_steps", 8)
    set_if_present(settings, "seed", 123456)
    result = SESSION.submit_task(settings).result(timeout=None)
    if not result.success:
        errs = [f"stage={getattr(e, 'stage', None)}: {getattr(e, 'message', str(e))}" for e in (result.errors or [])]
        for line in errs:
            print("SMOKE ERROR:", line, flush=True)
        raise RuntimeError("\n".join(errs) or "Startup smoke test failed")
    files = [str(Path(p).resolve()) for p in (result.generated_files or []) if Path(p).exists()]
    SMOKE_OUTPUT = pick_primary(files)
    if not SMOKE_OUTPUT:
        raise RuntimeError("Startup smoke test produced no media file")


def initialize() -> None:
    global SESSION, READY, STARTUP_ERROR
    try:
        SESSION = init(
            root=str(WAN_ROOT),
            output_dir=str(OUTPUT_DIR),
            cli_args=(
                "--profile", "4",
                "--attention", "sdpa",
                "--vram-allocator", "vmm_spill",
                "--perc-reserved-mem-max", "0.45",
                "--verbose", "1",
            ),
            console_output=True,
            console_isatty=False,
        )
        for model_type in (MODEL_TYPE_FL2VA, MODEL_TYPE_REF2VA):
            if model_type and SESSION.get_model_def(model_type) is None:
                raise RuntimeError(f"Model definition not found: {model_type}")
        startup_smoke()
        READY = True
    except Exception as exc:
        STARTUP_ERROR = f"{type(exc).__name__}: {exc}"
        READY = False
        raise


app = FastAPI(title="MiniMax H3 Kaggle API", version="1.0.0", docs_url=None, redoc_url=None)


@app.get("/health")
async def health() -> JSONResponse:
    body = {
        "status": "ready" if READY else "starting" if STARTUP_ERROR is None else "failed",
        "ready": READY,
        "model": "MiniMax H3",
        "models": {"fl2va": MODEL_TYPE_FL2VA, "ref2va": MODEL_TYPE_REF2VA or None},
        "quality_defaults": {
            "resolution": DEFAULT_RESOLUTION,
            "duration_seconds": DEFAULT_DURATION,
            "steps": DEFAULT_STEPS,
            "turbo": False,
            "step_caches": False,
            "attention": "sdpa",
            "profile": 4,
            "quality": "maximum",
        },
        "gpu": gpu_snapshot(),
        "smoke_test_output": SMOKE_OUTPUT,
        "error": STARTUP_ERROR,
    }
    return JSONResponse(body, status_code=200 if READY else 503 if STARTUP_ERROR else 200)


@app.get("/v1/models")
async def models(request: Request) -> JSONResponse:
    require_auth(request)
    data = []
    for public_name, model_type in (("fl2va", MODEL_TYPE_FL2VA), ("ref2va", MODEL_TYPE_REF2VA)):
        if not model_type:
            continue
        metadata = SESSION.get_model_metadata(model_type, include_availability=True) or {}
        data.append({
            "id": public_name,
            "model_type": model_type,
            "name": metadata.get("name", model_type),
            "description": metadata.get("description", ""),
            "availability": metadata.get("availability"),
        })
    return JSONResponse({"data": data})


@app.get("/v1/system")
async def system_info(request: Request) -> JSONResponse:
    require_auth(request)
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage(OUTPUT_DIR)
    return JSONResponse({
        "gpu": gpu_snapshot(),
        "ram": {
            "total_gb": round(vm.total / 2**30, 2),
            "available_gb": round(vm.available / 2**30, 2),
        },
        "output_free_gb": round(disk.free / 2**30, 2),
        "active_job": ACTIVE_JOB_ID,
    })


@app.post("/v1/videos")
async def create_video(request: Request) -> JSONResponse:
    global ACTIVE_JOB_ID
    require_auth(request)
    if not READY:
        raise HTTPException(status_code=503, detail="H3 server is not ready")

    try:
        payload, media = await parse_input(request)
        model_type, settings = build_settings(payload, media)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid request: {type(exc).__name__}: {exc}") from exc

    with JOB_LOCK:
        if ACTIVE_JOB_ID:
            active = JOBS.get(ACTIVE_JOB_ID)
            raise HTTPException(
                status_code=409,
                detail={
                    "message": "A quality-first H3 generation is already running.",
                    "active_job_id": ACTIVE_JOB_ID,
                    "active_status": active.status if active else "running",
                },
            )
        job_id = f"h3_{uuid.uuid4().hex[:16]}"
        JOBS[job_id] = JobRecord(
            id=job_id,
            status="queued",
            created_at=utc_now(),
            request=payload,
            model_type=model_type,
        )
        ACTIVE_JOB_ID = job_id

    threading.Thread(target=run_job, args=(job_id, settings), daemon=True).start()
    return JSONResponse({
        "id": job_id,
        "status": "queued",
        "model": payload.get("model", "fl2va"),
        "model_type": model_type,
        "created_at": JOBS[job_id].created_at,
    }, status_code=202)


@app.get("/v1/videos/{job_id}")
async def get_video(job_id: str, request: Request) -> JSONResponse:
    require_auth(request)
    record = JOBS.get(job_id)
    if not record:
        raise HTTPException(status_code=404, detail="Unknown job id")
    return JSONResponse({
        "id": record.id,
        "status": record.status,
        "created_at": record.created_at,
        "started_at": record.started_at,
        "finished_at": record.finished_at,
        "model_type": record.model_type,
        "download_url": f"/v1/videos/{record.id}/download" if record.primary_file else None,
        "generated_files": [Path(x).name for x in (record.generated_files or [])],
        "error": record.error,
    })


@app.get("/v1/videos/{job_id}/download")
async def download_video(job_id: str, request: Request) -> FileResponse:
    require_auth(request)
    record = JOBS.get(job_id)
    if not record:
        raise HTTPException(status_code=404, detail="Unknown job id")
    if record.status != "succeeded" or not record.primary_file:
        raise HTTPException(status_code=409, detail="Job is not completed successfully")
    path = Path(record.primary_file)
    if not path.exists():
        raise HTTPException(status_code=410, detail="Generated file is no longer available")
    return FileResponse(path, filename=path.name, media_type="application/octet-stream")


@app.post("/v1/videos/{job_id}/cancel")
async def cancel_video(job_id: str, request: Request) -> JSONResponse:
    require_auth(request)
    record = JOBS.get(job_id)
    if not record:
        raise HTTPException(status_code=404, detail="Unknown job id")
    active = getattr(SESSION, "_active_job", None)
    if active is not None:
        try:
            active.cancel()
        except Exception:
            pass
    record.status = "cancel_requested"
    return JSONResponse({"id": job_id, "status": record.status})


if __name__ == "__main__":
    initialize()
    uvicorn.run(app, host=HOST, port=PORT, log_level="info")
'''
SERVER_PY.write_text(SERVER_SOURCE, encoding="utf-8")
print("Server written to:", SERVER_PY)


In [ ]:

# 8) Discover the exact generated model types and validate the H3 definitions.
# Imports WanGP's API but does not start Gradio.

import sys, json
sys.path.insert(0, str(WAN_ROOT))

from shared.api import init as wangp_init

probe_session = wangp_init(
    root=str(WAN_ROOT),
    output_dir=str(OUTPUT_ROOT),
    cli_args=(
        "--profile", "4",
        "--attention", "sdpa",
        "--vram-allocator", "vmm_spill",
        "--perc-reserved-mem-max", "0.45",
        "--verbose", "1",
    ),
    console_output=False,
    console_isatty=False,
)

h3_models = probe_session.list_model_metadata(family="minimax_h3")
print("H3-related model definitions visible to the API:")
for m in h3_models:
    print("-", m.get("model_type"), "=>", m.get("name"))

fl2va_matches = [m for m in h3_models if "fl2va" in str(m.get("model_type", "")).lower() and ("h3_kaggle" in str(m.get("model_type", "")).lower() or "pruned rank-8" in str(m.get("name", "")).lower())]
ref_matches = [m for m in h3_models if "ref2va" in str(m.get("model_type", "")).lower() and "h3_kaggle" in str(m.get("model_type", "")).lower()]

if not fl2va_matches:
    raise RuntimeError("Our pinned FL2VA model definition was not discovered by WanGP.")

H3_MODEL_TYPE_FL2VA = fl2va_matches[0]["model_type"]
H3_MODEL_TYPE_REF2VA = ref_matches[0]["model_type"] if ref_matches else ""

print("\nSelected FL2VA:", H3_MODEL_TYPE_FL2VA)
print("Selected Ref2VA:", H3_MODEL_TYPE_REF2VA or "(not available)")



## 8b) Pre-download all required model assets

We download the exact Hugging Face files into the same layout WanGP expects
under `/tmp/Wan2GP/ckpts/` using the normal `hf_hub_download` path directly,
so the server never has to use WanGP's patched HTTP downloader.


In [ ]:

# 8b) Stage assets — prefer an attached Kaggle dataset; fall back to Hugging Face.

import os, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

CKPT = WAN_ROOT / "ckpts"
CKPT.mkdir(parents=True, exist_ok=True)

# 1) If a Kaggle dataset is mounted, copy matching files into the WanGP layout.
for base in [Path("/kaggle/input")]:
    for root, _, files in os.walk(base):
        for f in files:
            f_low = f.lower()
            if "minimax-h3" not in f_low and "qwen3-vl" not in f_low and "minimax_h3" not in f_low:
                continue
            if "qwen3-vl" in f_low:
                dst = CKPT / "Qwen3-VL-32B-Instruct" / f
            elif f.startswith("minimax_h3"):
                dst = CKPT / "minimax_h3" / f
            else:
                dst = CKPT / f
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists() and Path(root, f).stat().st_size > 0:
                print("stage from dataset:", f, "->", dst)
                shutil.copy2(Path(root, f), dst)

ASSETS = [
    "MiniMax-H3-FL2VA-pruned_rank8_int8_convrot.safetensors",
    "MiniMax-H3-video_vae_fp16.safetensors",
    "MiniMax-H3-audio_vae_fp32.safetensors",
    "minimax_h3/minimax_h3_latent_upscaler_3d_bf16.safetensors",
    "minimax_h3/MiniMax-H3-X2-Detail-v1.safetensors",
    "Qwen3-VL-32B-Instruct/config.json",
    "Qwen3-VL-32B-Instruct/tokenizer_config.json",
    "Qwen3-VL-32B-Instruct/tokenizer.json",
    "Qwen3-VL-32B-Instruct/vocab.json",
    "Qwen3-VL-32B-Instruct/preprocessor_config.json",
    "Qwen3-VL-32B-Instruct/chat_template.json",
    "Qwen3-VL-32B-Instruct/qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf",
]
for a in ASSETS:
    dest = CKPT / a
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 1024:
        print("exists/staged:", a)
        continue
    p = hf_hub_download(repo_id=HF_REPO, filename=a, local_dir=str(CKPT))
    print("downloaded:", a, "->", p)
print("Assets ready under:", CKPT)


In [ ]:

# 9) Start the FastAPI process in its own session/process.

import os, subprocess, time, requests, json, signal

server_env = os.environ.copy()
server_env.update({
    "WAN2GP_ROOT": str(WAN_ROOT),
    "H3_OUTPUT_DIR": str(OUTPUT_ROOT),
    "H3_UPLOAD_DIR": str(UPLOAD_ROOT),
    "H3_API_PORT": str(API_PORT),
    "H3_API_HOST": API_HOST,
    "H3_API_KEY": API_KEY,
    "H3_MODEL_TYPE_FL2VA": H3_MODEL_TYPE_FL2VA,
    "H3_MODEL_TYPE_REF2VA": H3_MODEL_TYPE_REF2VA,
    "H3_DEFAULT_RESOLUTION": DEFAULT_RESOLUTION,
    "H3_DEFAULT_STEPS": str(DEFAULT_STEPS),
    "H3_DEFAULT_DURATION": str(DEFAULT_DURATION_SECONDS),
    "H3_DEFAULT_SEED": str(DEFAULT_SEED),
    "H3_STARTUP_SMOKE_TEST": "1" if STARTUP_SMOKE_TEST else "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "HF_HUB_DISABLE_XET": "1",
    "HF_HUB_DISABLE_HF_TRANSFER": "1",
    "HF_HOME": os.environ["HF_HOME"],
    "HF_HUB_CACHE": os.environ["HF_HUB_CACHE"],
    "TORCH_HOME": os.environ["TORCH_HOME"],
    "XDG_CACHE_HOME": os.environ["XDG_CACHE_HOME"],
})

server_log = LOG_ROOT / "server.log"
server_out = server_log.open("w", encoding="utf-8")

server_proc = subprocess.Popen(
    [sys.executable, str(SERVER_PY)],
    cwd=str(WAN_ROOT),
    env=server_env,
    stdout=server_out,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Started FastAPI server PID:", server_proc.pid)
print("Server log:", server_log)


In [ ]:

# 10) Wait for the real H3 smoke test to finish.

health_url = f"http://127.0.0.1:{API_PORT}/health"
local_health = None

for i in range(360):  # up to 60 minutes
    try:
        r = requests.get(health_url, timeout=10)
        if r.status_code == 200:
            local_health = r.json()
            break
        if r.status_code == 503:
            data = r.json()
            if data.get("status") == "failed":
                print(json.dumps(data, indent=2))
                raise RuntimeError("H3 server initialization failed. See server.log.")
    except requests.RequestException:
        pass

    if i % 10 == 0:
        print(f"Waiting for H3 server... {i*10}s")
        if server_proc.poll() is not None:
            print("Server exited. Last log tail:")
            print(server_log.read_text(encoding="utf-8", errors="replace")[-6000:])
            raise RuntimeError("FastAPI process exited during startup.")
    time.sleep(10)

if local_health is None:
    print(server_log.read_text(encoding="utf-8", errors="replace")[-12000:])
    raise TimeoutError("H3 server did not become ready within the startup window.")

print(json.dumps(local_health, indent=2))


In [ ]:

# 11) Start a Cloudflare Quick Tunnel and verify the PUBLIC URL against /health.

cloudflare_log = LOG_ROOT / "cloudflared.log"
cf_out = cloudflare_log.open("w", encoding="utf-8")

cf_proc = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{API_PORT}"],
    stdout=cf_out,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Started cloudflared PID:", cf_proc.pid)

public_url = None
pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")

for i in range(180):
    text = cloudflare_log.read_text(encoding="utf-8", errors="replace")
    match = pattern.search(text)
    if match:
        candidate = match.group(0)
        try:
            rr = requests.get(candidate + "/health", timeout=15)
            if rr.status_code == 200 and rr.json().get("ready") is True:
                public_url = candidate
                break
        except requests.RequestException:
            pass

    if i % 5 == 0:
        print(f"Waiting for Cloudflare Quick Tunnel... {i*2}s")
    time.sleep(2)

if not public_url:
    print(cloudflare_log.read_text(encoding="utf-8", errors="replace")[-8000:])
    raise RuntimeError("Cloudflare Quick Tunnel did not become publicly healthy.")

print("Verified public endpoint:", public_url)


In [ ]:

# 12) FINAL READY BLOCK — printed only after runtime + smoke test + FastAPI + public tunnel are all healthy.

print("\n" + "="*76)
print("                 MINIMAX H3 API — READY")
print("="*76)
print(f"Base URL:")
print(f"  {public_url}")
print()
print("API Key:")
print(f"  {API_KEY}")
print()
print("Model:")
print("  MiniMax H3 FL2VA — Pruned Rank-8 INT8 ConvRot")
print("  Qwen3-VL 32B — Q4_K_M GGUF")
print()
print("Quality defaults:")
print(f"  Resolution : {DEFAULT_RESOLUTION}")
print(f"  Duration   : {DEFAULT_DURATION_SECONDS}s")
print(f"  Steps      : {DEFAULT_STEPS}")
print("  Turbo      : OFF")
print("  Caches     : OFF")
print("  Attention  : SDPA (T4-safe)")
print("  Profile    : 4")
print()
print("Health:")
print(f"  {public_url}/health")
print()
print("Use locally:")
print("  export H3_BASE_URL='" + public_url + "'")
print("  export H3_API_KEY='" + API_KEY + "'")
print()
print("Then:")
print("  python h3_cli.py generate --wait \\")
print('    --prompt "A cinematic ..." \\')
print("    --resolution 1344x768 --duration 5 --steps 28 \\")
print("    --output result.mp4")
print()
print("Stop the Kaggle session when you are done; the API and Quick Tunnel disappear.")
print("="*76)


In [ ]:
# 13) Local API client CLI.
CLI_PY = API_ROOT / "h3_cli.py"
CLI_SOURCE = r'''#!/usr/bin/env python3
"""Local CLI for the MiniMax H3 Kaggle API server."""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

import requests


def env(name: str, default: str = "") -> str:
    return os.environ.get(name, default)


def headers(key: str) -> dict[str, str]:
    return {"Authorization": f"Bearer {key}"}


def cmd_health(args) -> int:
    r = requests.get(f"{args.base_url}/health", timeout=30)
    print(json.dumps(r.json(), indent=2))
    return 0 if r.status_code == 200 else 1


def cmd_generate(args) -> int:
    payload = {
        "model": args.model,
        "prompt": args.prompt,
        "resolution": args.resolution,
        "duration_seconds": args.duration,
        "steps": args.steps,
        "seed": args.seed,
        "quality": "maximum",
    }
    if args.sampler:
        payload["sampler"] = args.sampler

    files = []
    form_files = {}
    for field, path in (
        ("image_start", args.image_start),
        ("image_end", args.image_end),
        ("video_guide", args.video_guide),
        ("audio_guide", args.audio_guide),
    ):
        if path:
            p = Path(path)
            form_files[field] = (p.name, p.open("rb"), "application/octet-stream")
            files.append(form_files[field][1])
    for ref in args.reference_image or []:
        p = Path(ref)
        fh = p.open("rb")
        files.append(fh)
        form_files.setdefault("reference_image", []).append((p.name, fh, "application/octet-stream"))

    try:
        if form_files and any(field in form_files for field in ("image_start", "image_end", "video_guide", "audio_guide")) or form_files.get("reference_image"):
            # flatten: requests wants list of tuples for repeated fields
            multipart = []
            for field, value in form_files.items():
                if isinstance(value, list):
                    for v in value:
                        multipart.append((field, v))
                else:
                    multipart.append((field, value))
            r = requests.post(
                f"{args.base_url}/v1/videos",
                headers=headers(args.api_key),
                data={"payload": json.dumps(payload)},
                files=multipart,
                timeout=120,
            )
        else:
            r = requests.post(
                f"{args.base_url}/v1/videos",
                headers={**headers(args.api_key), "Content-Type": "application/json"},
                data=json.dumps(payload),
                timeout=120,
            )
    finally:
        for fh in files:
            try:
                fh.close()
            except Exception:
                pass

    if r.status_code not in (200, 202):
        print(f"Error {r.status_code}: {r.text}", file=sys.stderr)
        return 1
    job = r.json()
    print(json.dumps(job, indent=2))
    if not args.wait:
        return 0

    job_id = job["id"]
    while True:
        time.sleep(args.poll)
        s = requests.get(f"{args.base_url}/v1/videos/{job_id}", headers=headers(args.api_key), timeout=30)
        info = s.json()
        print(f"[{info.get('status')}]")
        if info.get("status") in ("succeeded", "failed", "cancel_requested"):
            break
    if info.get("status") != "succeeded":
        print(json.dumps(info, indent=2), file=sys.stderr)
        return 1

    out = Path(args.output)
    with requests.get(
        f"{args.base_url}/v1/videos/{job_id}/download", headers=headers(args.api_key), stream=True, timeout=600
    ) as dl:
        dl.raise_for_status()
        with out.open("wb") as f:
            for chunk in dl.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved: {out.resolve()}")
    return 0


def main() -> int:
    ap = argparse.ArgumentParser(prog="h3_cli.py", description="MiniMax H3 API client")
    sub = ap.add_subparsers(dest="command", required=True)

    for name in ("health",):
        p = sub.add_parser(name)
        p.add_argument("--base-url", default=env("H3_BASE_URL"))
        p.add_argument("--api-key", default=env("H3_API_KEY"))

    g = sub.add_parser("generate")
    g.add_argument("--base-url", default=env("H3_BASE_URL"))
    g.add_argument("--api-key", default=env("H3_API_KEY"))
    g.add_argument("--model", default="fl2va", choices=["fl2va", "ref2va"])
    g.add_argument("--prompt", required=True)
    g.add_argument("--resolution", default="1344x768")
    g.add_argument("--duration", type=float, default=5.0)
    g.add_argument("--steps", type=int, default=28)
    g.add_argument("--seed", type=int, default=-1)
    g.add_argument("--sampler", default=None, choices=[None, "euler", "res_multistep", "ralston_2s"])
    g.add_argument("--image-start", default=None)
    g.add_argument("--image-end", default=None)
    g.add_argument("--video-guide", default=None)
    g.add_argument("--audio-guide", default=None)
    g.add_argument("--reference-image", action="append", default=None)
    g.add_argument("--wait", action="store_true")
    g.add_argument("--poll", type=float, default=15.0)
    g.add_argument("--output", default="result.mp4")

    args = ap.parse_args()
    if not args.base_url:
        print("Set --base-url or H3_BASE_URL", file=sys.stderr)
        return 2
    if getattr(args, "api_key", "") == "" and args.command != "health":
        print("Set --api-key or H3_API_KEY", file=sys.stderr)
        return 2

    if args.command == "health":
        return cmd_health(args)
    if args.command == "generate":
        return cmd_generate(args)
    return 2


if __name__ == "__main__":
    raise SystemExit(main())
'''
CLI_PY.write_text(CLI_SOURCE, encoding="utf-8")
print("CLI written to:", CLI_PY)



## 14) Local API examples

### Health

```bash
curl "$H3_BASE_URL/health"
```

### Submit a generation

```bash
curl -X POST "$H3_BASE_URL/v1/videos" \
  -H "Authorization: Bearer $H3_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "fl2va",
    "prompt": "A cinematic single-take shot of a futuristic Indian railway station at dawn, realistic people, physically coherent motion, warm volumetric light, slow dolly forward. overall_soundscape: distant train, footsteps, station ambience.",
    "resolution": "1344x768",
    "duration_seconds": 5,
    "steps": 28,
    "seed": 12345,
    "quality": "maximum"
  }'
```

### Poll

```bash
curl "$H3_BASE_URL/v1/videos/JOB_ID" -H "Authorization: Bearer $H3_API_KEY"
```

### Download

```bash
curl -L "$H3_BASE_URL/v1/videos/JOB_ID/download" -H "Authorization: Bearer $H3_API_KEY" -o result.mp4
```

### Local CLI

```bash
python h3_cli.py generate \
  --base-url "$H3_BASE_URL" --api-key "$H3_API_KEY" \
  --prompt "A cinematic..." --resolution 1344x768 --duration 5 --steps 28 \
  --wait --output result.mp4
```

## Quality-first notes

- 28 → 32 steps, Euler → RES Multistep, longer durations via H3 sliding windows,
  FL2VA start/end images, Ref2VA reference images for identity control.
- Do **not** enable Turbo/TeaCache/Spectrum/First-Block Cache for this quality-first path.

## Safety / networking

Quick Tunnel URLs are public; keep the API key secret. Quick Tunnels use polling because SSE is not supported.


In [ ]:

# OPTIONAL: stop the API/tunnel without closing the whole Kaggle session.
# Guarded so it does not run during a normal notebook execution; set H3_SHUTDOWN=1 before running this cell to force stop.
import os, signal

SHUTDOWN = os.environ.get("H3_SHUTDOWN", "0") == "1"
if SHUTDOWN:
    for proc_name in ("cf_proc", "server_proc"):
        proc = globals().get(proc_name)
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
                print("Stopped", proc_name, proc.pid)
            except Exception as exc:
                print("Could not stop", proc_name, ":", exc)
else:
    print("Skipping auto-shutdown. Set H3_SHUTDOWN=1 and re-run this cell to stop server/tunnel.")
